<a href="https://colab.research.google.com/github/InduwaraGayashan001/LangChain/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup

In [1]:
from google.colab import userdata

In [ ]:
!pip install langchain_huggingface langchain-community langchain-openai

# Loading Documents

In [40]:
from langchain_classic.document_loaders import TextLoader

loader = TextLoader("/content/romeo-and-juliet.txt")
documents = loader.load()


# Splitting Documents into Chunks

In [41]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)
len(chunks)

182

In [42]:
chunks[0]

Document(metadata={'source': '/content/romeo-and-juliet.txt'}, page_content="Romeo and Juliet\nby William Shakespeare\nEdited by Barbara A. Mowat and Paul Werstine\n  with Michael Poston and Rebecca Niles\nFolger Shakespeare Library\nhttps://shakespeare.folger.edu/shakespeares-works/romeo-and-juliet/\nCreated on Jul 31, 2015, from FDT version 0.9.2\n\nCharacters in the Play\n======================\nROMEO\nMONTAGUE, his father\nLADY MONTAGUE, his mother\nBENVOLIO, their kinsman\nABRAM, a Montague servingman\nBALTHASAR, Romeo's servingman\nJULIET\nCAPULET, her father\nLADY CAPULET, her mother\nNURSE to Juliet\nTYBALT, kinsman to the Capulets\nPETRUCHIO, Tybalt's companion\nCapulet's Cousin\nServingmen:\n  SAMPSON\n  GREGORY\n  PETER\nOther Servingmen\nESCALUS, Prince of Verona\nPARIS, the Prince's kinsman and Juliet's suitor\nMERCUTIO, the Prince's kinsman and Romeo's friend\nParis' Page\nFRIAR LAWRENCE\nFRIAR JOHN\nAPOTHECARY\nThree or four Citizens\nThree Musicians\nThree Watchmen\nCHO

# Creating Embeddings

In [43]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=userdata.get("GITHUB_TOKEN"),
    base_url="https://models.github.ai/inference"
)


# Storing Embeddings in a Vector Store

In [ ]:
!pip install faiss-cpu

In [44]:
from langchain_classic.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

# Retrieving Relevant Context

In [45]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
query = "How did Juliet die?"
retriever.invoke(query)

[Document(id='1266053b-5d0e-43c1-b842-c1c472021a3e', metadata={'source': '/content/romeo-and-juliet.txt'}, page_content="JULIET\nGo, get thee hence, for I will not away.\n[He exits.]\nWhat's here? A cup closed in my true love's hand?\nPoison, I see, hath been his timeless end.--\nO churl, drunk all, and left no friendly drop\nTo help me after! I will kiss thy lips.\nHaply some poison yet doth hang on them,\nTo make me die with a restorative.\t[She kisses him.]\nThy lips are warm!\n\n[Enter Paris's Page and Watch.]\n\n\nFIRST WATCH  Lead, boy. Which way?\n\nJULIET\nYea, noise? Then I'll be brief. O, happy dagger,\nThis is thy sheath. There rust, and let me die.\n[She takes Romeo's dagger, stabs herself, and dies.]\n\nPAGE\nThis is the place, there where the torch doth burn."),
 Document(id='63d17bf9-6ba2-4d2d-8691-2f629a6370fe', metadata={'source': '/content/romeo-and-juliet.txt'}, page_content='NURSE  Romeo can,\nThough heaven cannot. O Romeo, Romeo,\nWhoever would have thought it? Rom

# Setting Up the LLM

In [46]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=userdata.get("GITHUB_TOKEN"),
    base_url="https://models.github.ai/inference"
)

# Generating Answers Using Retrieved Context

In [47]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Use the following context to answer the question.\n\n"
        "Context:\n{context}\n\n"
        "Question:\n{question}"
    )
)

parser = StrOutputParser()

# Building the RAG Chain

In [48]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = prompt | llm | parser


# Running the RAG Pipeline

In [49]:
response = rag_chain.invoke({"context": format_docs(retriever.invoke(query)), "question": query})
print(response)

Juliet died by stabbing herself with Romeo's dagger. After discovering that Romeo had died from poison, she expressed her despair and, feeling unable to live without him, took the dagger and ended her own life.
